# Reddit Experiment Data Processing

Extracts data from the political signaling username reddit experiment and puts it into an inference-ready format.

In [1]:
from pathlib import Path
from pyprojroot import here

import json
import polars as pl

In [2]:
# Needed to filter out non-experimental posts
experiment_subreddits = [
    "Baking",
    "bodyweightfitness",
    "explainlikeimfive",
    "NoStupidQuestions",
    "AskDocs", 
    "cooking", 
    "TravelHacks", 
    "netflix", 
    "AskCulinary",
    "LearnProgramming", 
    "backpacking", 
    "camping", 
    "socialskills",
    "booksuggestions", 
    "Fantasy", 
    "Cycling", 
    "Sleep", 
    "CasualConversation",
    "AskMen", 
    "hiking", 
    "Houseplants", 
    "LanguageLearning", 
    "Photography",
    "frugal",
    "selfimprovement",
    "Pets",
    "Skiing", 
    "IWantToLearn",
    "Beauty", 
    "GiftIdeas"
]

In [9]:
experiment_results = Path(here("data/experiment_results"))
reply_schema = {
    'id': pl.String,
    'subreddit': pl.String,
    'username': pl.String,
    'username_score': pl.Int64,
    'label': pl.String,
    'content': pl.String
    }
replies = pl.DataFrame(schema=reply_schema)
posts = pl.DataFrame(schema={
    'id': pl.String,
    'title': pl.String,
    'post_text': pl.String
})

for user in experiment_results.iterdir():
    if not user.is_dir():
        continue

    username = user.name
    label = "liberal" if "no_kings" in username else "conversative" if "MAGA" in username else "neutral"
    username_score = -1 if label == "liberal" else 1 if label == "conversative" else 0

    for post in user.iterdir():
        if not post.is_dir():
            continue

        id, subreddit = [s[::-1] for s in post.name[::-1].split('_', maxsplit=1)]

        if subreddit not in experiment_subreddits:
            continue

        collect = post / "24h.json"
        if not collect.is_file():
            continue

        with collect.open("r") as f:
            collect_dict = json.load(f)

            new_post = pl.DataFrame({
                "id": id,
                "title": collect_dict["post"]["title"],
                "post_text": collect_dict["post"]["selftext"]
            })

            posts = pl.concat([posts, new_post])

            contents = [repr(comments["body"]) for comments in collect_dict["comments"]]
            if not contents:
                continue

            starter_dict = {
                "id": id,
                "subreddit": subreddit,
                "username": username,
                "username_score": username_score,
                "label": label
            }
            new_rows = pl.DataFrame([starter_dict] * len(contents))
            new_rows = new_rows.with_columns(pl.Series(contents).alias("content"))
            replies = pl.concat([replies, new_rows])

# Move label column to the end
replies = replies.select([pl.exclude("label"), "label"])

replies


id,subreddit,username,username_score,content,label
str,str,str,i64,str,str
"""1rwn60r""","""booksuggestions""","""Fresh_Window_6484""",0,"""""The Emperor's Assassin series…","""neutral"""
"""1rwn60r""","""booksuggestions""","""Fresh_Window_6484""",0,"""'You might like The Darktouche…","""neutral"""
"""1rvbenb""","""AskMen""","""Fresh_Window_6484""",0,"""'Your submission was removed b…","""neutral"""
"""1rpex9a""","""socialskills""","""Fresh_Window_6484""",0,"""'\nThanks for your post at /r/…","""neutral"""
"""1rpex9a""","""socialskills""","""Fresh_Window_6484""",0,"""'Be ok with having uncomfortab…","""neutral"""
…,…,…,…,…,…
"""1s0bn9y""","""camping""","""MAGA_victory24""",1,"""'If I am car camping I bring a…","""conversative"""
"""1s0bn9y""","""camping""","""MAGA_victory24""",1,"""'Burn a 12 hour candle inside …","""conversative"""
"""1s0bn9y""","""camping""","""MAGA_victory24""",1,"""'Tents vary a lot, so some of …","""conversative"""


In [10]:
replies.write_parquet(here("data/experiment.parquet"))

In [11]:
posts.write_parquet(here("data/experiment_posts.parquet"))